# Kaggle: Chuyển đổi VisDrone **MOT + DET** từ hai Kaggle Dataset

Notebook này dùng **mã nguồn gốc** từ [`ntrThanh/uav-tracking`](https://github.com/ntrThanh/uav-tracking) (pin commit `950379c`) để tạo **YOLO / RF-DETR / COCO / MOTChallenge** cùng một quy trình, không tự viết lại logic chuyển nhãn.

**Trước khi chạy:**

1. Trong Kaggle tạo Notebook mới, **Add Input cả hai Dataset**: (1) dataset MOT cũ, (2) dataset DET mới.
2. Bật **Internet** trong Notebook Settings (để clone GitHub).
3. Chạy lần lượt từ đầu đến cuối. Không cần GPU cho bước chuyển định dạng.
4. Kết quả: **một ZIP tự chứa** tại `/kaggle/working/visdrone_mot_det_converted.zip` để đưa lên Kaggle Dataset mới.

**Input được hỗ trợ:** thư mục MOT đã giải nén như `/kaggle/input/.../VisDrone/Task4_MOT/VisDrone2019-MOT-train/`; DET là thư mục đã giải nén, ba ZIP riêng, hoặc ZIP ngoài `VisDrone_DET.zip` chứa ba ZIP con. **Không dùng** `Task2_VID` và `test-challenge` (không có nhãn công khai).

**Quan trọng:** ZIP cuối có cả ảnh JPEG gốc một lần + các file nhãn được chuyển đổi + `restore_links.py`, nên file sẽ khá lớn. Kaggle `/kaggle/input` chỉ đọc, `/kaggle/working` dùng để giải nén DET và ghi đầu ra. Để chạy train ở notebook khác, giải nén ZIP vào thư mục ghi được rồi chạy `restore_links.py`.


In [ ]:
from pathlib import Path
import os, re, sys, json, shutil, subprocess, zipfile, time


# CẤU HÌNH: Add Input CẢ HAI Kaggle Dataset trước khi chạy
KAGGLE_INPUT = Path('/kaggle/input')
REPO_URL = 'https://github.com/ntrThanh/uav-tracking.git'
REPO_REF = '950379c83ce853c8dce1d0110907a0753cc6f007'  # pin phiên bản đã kiểm tra

WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'uav-tracking'
EXTRACT_DIR = WORK / 'visdrone_det_extracted'
OUTPUT_ZIP = WORK / 'visdrone_mot_det_converted.zip'

# Repo hỗ trợ mot / det / motdet. Mặc định tạo đủ cả 3.
VARIANTS = ['mot', 'motdet', 'det']
RECOMPUTE_DET_OVERLAP = False  # dùng danh sách lọc 57 ảnh có sẵn trong repo
DRAW_CHECK_IMAGES = 0          # xuất ảnh kiểm tra nhãn
CLEANUP_WORKING = True         # xóa repo/temp sau khi ZIP đã kiểm tra thành công

assert KAGGLE_INPUT.is_dir(), f'Không có {KAGGLE_INPUT}: hãy chạy trên Kaggle.'
WORK.mkdir(parents=True, exist_ok=True)
print('Kaggle inputs:', [p.name for p in KAGGLE_INPUT.iterdir() if p.is_dir()])
print('Free disk (GiB):', round(shutil.disk_usage(WORK).free / 1024**3, 2))


Kaggle inputs: ['datasets']
Free disk (GiB): 19.5


## 1. Kiểm tra **hai** Kaggle Dataset

Notebook tìm toàn bộ thư mục đã giải nén ở `/kaggle/input`, không phụ thuộc tên slug của dataset. MOT được giữ nguyên ở Kaggle Input (không copy ảnh). DET chỉ được giải nén vào `/kaggle/working` khi Kaggle chưa tự giải nén ZIP.


In [2]:
print('Kaggle Dataset đã Add Input:')
for p in sorted(KAGGLE_INPUT.iterdir()):
    print(' -', p.name, '[folder]' if p.is_dir() else '[file]')
print('Input chỉ được đọc, không bị chỉnh sửa.')


Kaggle Dataset đã Add Input:
 - datasets [folder]
Input chỉ được đọc, không bị chỉnh sửa.


## 2. Lấy đúng mã nguồn từ repository

In [3]:
if not (REPO_DIR / '.git').is_dir():
    print('Cloning', REPO_URL)
    try:
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Không clone được GitHub. Hãy bật Internet cho Kaggle Notebook hoặc '
            'thêm code repo dưới dạng Kaggle Input rồi điều chỉnh REPO_DIR.'
        ) from exc

# Giữ các script gốc từ GitHub; không copy/paste hoặc sửa thuật toán chuyển đổi.
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--quiet', REPO_REF], check=True)
REPO_SHA = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
for rel in ('data/tools/prepare.py', 'data/tools/visdrone.py',
            'data/tools/check.py', 'data/tools/det_overlap.py'):
    assert (REPO_DIR / rel).is_file(), f'Thiếu script repo: {rel}'
print('Repo SHA:', REPO_SHA)
print('Script chính:', REPO_DIR / 'data/tools/prepare.py')

Cloning https://github.com/ntrThanh/uav-tracking.git
Repo SHA: 950379c83ce853c8dce1d0110907a0753cc6f007
Script chính: /kaggle/working/uav-tracking/data/tools/prepare.py


## 3. Tìm MOT đã giải nén, tìm hoặc giải nén DET

Tìm đủ ba split `train`, `val`, `test-dev` cho mỗi loại. Nếu DET đang nằm trong một ZIP ngoài chứa ba ZIP con (ví dụ `VisDrone_DET.zip`), chỉ giải nén **ba ZIP cần thiết**; bỏ `DET-test-challenge`.


In [4]:
NAME_RE = re.compile(r'^VisDrone2019-(MOT|DET)-(train|val|test-dev)$', re.IGNORECASE)
SPLITS = ('train', 'val', 'test-dev')
REQUIRED = {f'VisDrone2019-{kind}-{split}' for kind in ('MOT', 'DET') for split in SPLITS}

def canonical_name(name: str):
    m = NAME_RE.fullmatch(Path(name).stem if str(name).lower().endswith('.zip') else str(name))
    return f'VisDrone2019-{m.group(1).upper()}-{m.group(2).lower()}' if m else None

def valid_source(p: Path, key: str):
    required = ('sequences', 'annotations') if '-MOT-' in key else ('images', 'annotations')
    return all((p / d).is_dir() for d in required)

def find_directories(root: Path):
    found = {}
    if not root.exists():
        return found
    for base, dirs, _files in os.walk(root):
        # Không quét lần lượt hàng vạn file ảnh.
        dirs[:] = [d for d in dirs if d not in ('images', 'annotations', 'sequences', '__MACOSX')]
        for d in list(dirs):
            key = canonical_name(d)
            if not key:
                continue
            candidate = Path(base) / d
            if valid_source(candidate, key):
                found.setdefault(key, candidate)
                dirs.remove(d)
    return found

def safe_unzip(archive: Path, target_dir: Path):
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        info = z.infolist()
        bytes_need = sum(i.file_size for i in info)
        free = shutil.disk_usage(WORK).free
        print(f'Giải nén {archive.name}: {bytes_need/1024**3:.2f} GiB / free {free/1024**3:.2f} GiB')
        if bytes_need > free * 0.92:
            raise RuntimeError('Không đủ dung lượng working disk để giải nén DET.')
        root = target_dir.resolve()
        for i in info:
            member = i.filename.replace('\\', '/')
            if not (target_dir / member).resolve().is_relative_to(root):
                raise ValueError(f'ZIP chứa đường dẫn không an toàn: {member}')
        z.extractall(target_dir)

def recover_extracted(key, destination):
    candidates = find_directories(destination)
    if key in candidates:
        return candidates[key]
    # Một số ZIP VisDrone chứa trực tiếp images/ + annotations/, không có folder tên split.
    if valid_source(destination, key):
        return destination
    raise RuntimeError(f'Đã giải nén nhưng không thấy {key} trong {destination}')

sources = find_directories(KAGGLE_INPUT)
print('Các thư mục dữ liệu có sẵn:', sorted(sources))

# Liệt kê ZIP ở các cấp bên ngoài ảnh; không mở ZIP MOT vì MOT đã có thư mục.
archive_paths = []
for base, dirs, filenames in os.walk(KAGGLE_INPUT):
    dirs[:] = [d for d in dirs if d not in ('images', 'annotations', 'sequences')]
    for f in filenames:
        if f.lower().endswith('.zip'):
            archive_paths.append(Path(base) / f)
archive_paths.sort()

# A. Tìm các ZIP DET riêng đã được Kaggle giải nén từ ZIP ngoài.
for arc in archive_paths:
    key = canonical_name(arc.name)
    if not key or '-DET-' not in key or key in sources:
        continue
    dest = EXTRACT_DIR / key
    safe_unzip(arc, dest)
    sources[key] = recover_extracted(key, dest)

# B. Nếu còn thiếu DET và chỉ có ZIP ngoài, đọc ZIP ngoài rồi lấy đúng ba ZIP con.
missing_det = {k for k in REQUIRED - sources.keys() if '-DET-' in k}
if missing_det:
    candidates = [p for p in archive_paths if 'det' in p.name.lower() and canonical_name(p.name) is None]
    for outer_zip in candidates:
        if not missing_det:
            break
        with zipfile.ZipFile(outer_zip) as outer:
            for member in outer.namelist():
                key = canonical_name(Path(member).name)
                if not key or key not in missing_det or not member.lower().endswith('.zip'):
                    continue
                # Stream ZIP con ra tạm, không giữ toàn bộ ZIP ngoài trong working.
                temporary = EXTRACT_DIR / f'{key}.zip'
                temporary.parent.mkdir(parents=True, exist_ok=True)
                with outer.open(member) as src, temporary.open('wb') as dst:
                    shutil.copyfileobj(src, dst, length=4*1024*1024)
                try:
                    dest = EXTRACT_DIR / key
                    safe_unzip(temporary, dest)
                    sources[key] = recover_extracted(key, dest)
                    missing_det.remove(key)
                finally:
                    temporary.unlink(missing_ok=True)

print('\nNguồn đã phát hiện:')
for key, directory in sorted(sources.items()):
    print(' ', key, '->', directory)
missing = sorted(REQUIRED - sources.keys())
assert not missing, (
    f'Thiếu {missing}. Hãy Add Input cả dataset MOT và DET, '
    'hoặc kiểm tra lại cấu trúc file ZIP của DET.'
)


Các thư mục dữ liệu có sẵn: ['VisDrone2019-DET-test-dev', 'VisDrone2019-DET-train', 'VisDrone2019-DET-val', 'VisDrone2019-MOT-test-dev', 'VisDrone2019-MOT-train', 'VisDrone2019-MOT-val']

Nguồn đã phát hiện:
  VisDrone2019-DET-test-dev -> /kaggle/input/datasets/tiendq/visdrone-det/VisDrone/Task1_DET/VisDrone2019-DET-test-dev
  VisDrone2019-DET-train -> /kaggle/input/datasets/tiendq/visdrone-det/VisDrone/Task1_DET/VisDrone2019-DET-train/VisDrone2019-DET-train
  VisDrone2019-DET-val -> /kaggle/input/datasets/tiendq/visdrone-det/VisDrone/Task1_DET/VisDrone2019-DET-val/VisDrone2019-DET-val
  VisDrone2019-MOT-test-dev -> /kaggle/input/datasets/tiendq/visdrone-dataset/VisDrone/Task4_MOT/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev
  VisDrone2019-MOT-train -> /kaggle/input/datasets/tiendq/visdrone-dataset/VisDrone/Task4_MOT/VisDrone2019-MOT-train/VisDrone2019-MOT-train
  VisDrone2019-MOT-val -> /kaggle/input/datasets/tiendq/visdrone-dataset/VisDrone/Task4_MOT/VisDrone2019-MOT-val/VisDr

## 4. Kiểm tra định dạng nhãn và nối các thư mục gốc vào repo

`data/raw` chỉ chứa symlink trỏ tới ảnh gốc trên Kaggle Input (MOT) hoặc thư mục đã giải nén (DET). Không copy MOT sang `/kaggle/working`.


In [5]:
def validate_original_annotations(key: str, src: Path):
    required = ('sequences', 'annotations') if '-MOT-' in key else ('images', 'annotations')
    assert all((src / d).is_dir() for d in required), f'{key} thiếu thư mục ảnh/annotation: {src}'
    labels = sorted((src / 'annotations').glob('*.txt'))
    assert labels, f'{key}: không tìm thấy nhãn gốc TXT'
    first = next((line for p in labels[:12] for line in p.read_text(errors='replace').splitlines() if line.strip()), None)
    assert first, f'{key}: chưa thấy dòng annotation'
    expected = 10 if '-MOT-' in key else 8
    cols = first.rstrip(',').split(',')
    assert len(cols) >= expected, f'{key}: cần {expected} cột annotation, thấy {len(cols)}'
    try:
        for v in cols[:expected]:
            int(v.strip())
    except ValueError as exc:
        raise ValueError(f'{key}: annotation không đúng dạng VisDrone gốc: {first[:100]}') from exc
    print(f'OK {key}: {len(labels):,} file annotation')

raw_dir = REPO_DIR / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)
for key in sorted(REQUIRED):
    source = sources[key]
    validate_original_annotations(key, source)
    link = raw_dir / key
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise FileExistsError(f'Không ghi đè {link}')
    link.symlink_to(source.resolve(), target_is_directory=True)
    assert link.is_dir()

print('Đã gắn 6 split gốc vào:', raw_dir)


OK VisDrone2019-DET-test-dev: 1,610 file annotation
OK VisDrone2019-DET-train: 6,471 file annotation
OK VisDrone2019-DET-val: 548 file annotation
OK VisDrone2019-MOT-test-dev: 17 file annotation
OK VisDrone2019-MOT-train: 56 file annotation
OK VisDrone2019-MOT-val: 7 file annotation
Đã gắn 6 split gốc vào: /kaggle/working/uav-tracking/data/raw


## 5. Chạy **code** và kiểm tra

Repo sẽ tạo cả ba biến thể: `mot` (MOT train stride 2), `motdet` (MOT stride 6 + DET train) và `det` (DET train/val/test-dev). Danh sách loại 57 ảnh DET trùng video MOT val/test-dev đã có sẵn trong repo.


In [6]:
def run_repo(*args):
    command = [sys.executable, *map(str, args)]
    print('\n$', ' '.join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, check=True)

if RECOMPUTE_DET_OVERLAP:
    run_repo('data/tools/det_overlap.py')
else:
    print('Dùng data/splits/det_train_excluded.txt đã có trong repo để lọc 57 ảnh.')

run_repo('data/tools/prepare.py', 'all', '--variants', *VARIANTS)
run_repo('data/tools/check.py', '--variants', *VARIANTS, '--draw', str(DRAW_CHECK_IMAGES))

processed = REPO_DIR / 'data' / 'processed'
print('\nOUTPUTS:')
for p in sorted(processed.iterdir()):
    print(' ', p.name)


Dùng data/splits/det_train_excluded.txt đã có trong repo để lọc 57 ảnh.

$ /usr/bin/python3 data/tools/prepare.py all --variants mot motdet det
[+] Raw data: /kaggle/working/uav-tracking/data/raw
[+] Variants: mot (MOT train stride 2), motdet (MOT train stride 6), det (MOT train stride 2)
    [-] MOT train: 56 sequences, 24,201 frames, 1,105,516 boxes
    [-] MOT val: 7 sequences, 2,846 frames, 114,132 boxes
    [-] MOT test-dev: 17 sequences, 6,635 frames, 307,799 boxes
    [-] DET train: 6,414 images (57 excluded), 338,285 boxes
    [-] DET val: 548 images, 38,759 boxes
    [-] DET test-dev: 1,610 images, 75,102 boxes
[+] [mot] Writing frame lists
    [-] splits/train.txt: 12,114 images (mot 12,114)
    [-] splits/val.txt: 2,846 images (mot 2,846)
    [-] splits/test-dev.txt: 6,635 images (mot 6,635)
[+] [mot] Writing COCO json (file_name relative to data/raw)
    [-] coco/train.json: 12,114 images, 553,391 boxes, 31,195 ignored regions (91 MB)
    [-] coco/val.json: 2,846 images, 11

## 6. Tạo ZIP tự chứa (không mất symlink)

Mỗi ảnh gốc được nén **một lần** trong `data/raw`; các symlink trong `data/processed` được ghi vào `links_manifest.json`. File `restore_links.py` khôi phục chúng **sau khi giải nén ZIP vào thư mục có quyền ghi**, đồng thời chỉnh `visdrone.yaml` sang đường dẫn tuyệt đối mới. Không cần chạy lại `prepare.py`.

**Lưu ý về dung lượng:** ZIP ảnh gốc MOT + DET khá lớn. Cần đủ dung lượng trống trong `/kaggle/working`; khi xuất, notebook bỏ qua dữ liệu `test-challenge` và VID vì không dùng cho training/evaluation hiện tại.


In [7]:
RESTORE_SCRIPT = r"""#!/usr/bin/env python3
# Khôi phục data/processed/* symlinks sau khi giải nén VisDrone bundle.
# Chạy: python restore_links.py (cùng thư mục data/).
from pathlib import Path
import json, os, re

ROOT = Path(__file__).resolve().parent
manifest = json.loads((ROOT/'links_manifest.json').read_text(encoding='utf-8'))
count = 0
for item in manifest['links']:
    link = ROOT / item['link']
    target = ROOT / item['target']
    if not target.exists():
        raise FileNotFoundError(f'Thiếu file/dir để khôi phục: {target}')
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise FileExistsError(f'Không ghi đè dữ liệu đang tồn tại: {link}')
    link.symlink_to(os.path.relpath(target, link.parent), target_is_directory=item['is_dir'])
    count += 1

# Sửa path: về thư mục thực tế sau khi unzip, để Ultralytics tìm ảnh đúng.
for yaml_file in (ROOT/'data'/'processed').glob('yolo*/visdrone.yaml'):
    raw = yaml_file.read_text(encoding='utf-8')
    new, n = re.subn(r'(?m)^path:\s*.*$', f'path: {yaml_file.parent}', raw)
    if n != 1:
        raise RuntimeError(f'Không sửa được trường path của {yaml_file}')
    yaml_file.write_text(new, encoding='utf-8')

print(f'Khôi phục {count:,} liên kết. YOLO YAML đã sửa path thành tuyệt đối.')
print('Ví dụ YOLO dataset:', ROOT/'data'/'processed'/'yolo'/'visdrone.yaml')
"""

def logical_symlink_target(link: Path) -> Path:
    # abspath/normpath KHÔNG dereference các symlink của data/raw đã gắn trên Kaggle.
    return Path(os.path.abspath(os.path.join(link.parent, os.readlink(link))))

links = []
for folder, dirs, filenames in os.walk(processed, followlinks=False):
    parent = Path(folder)
    for name in list(dirs):
        p = parent/name
        if p.is_symlink():
            dirs.remove(name)
            links.append(p)
    for name in filenames:
        p = parent/name
        if p.is_symlink():
            links.append(p)

link_manifest = []
for p in links:
    target = logical_symlink_target(p)
    if not target.is_relative_to(REPO_DIR):
        raise ValueError(f'Symlink ra ngoài repo, không thể portable: {p} -> {target}')
    if not target.is_relative_to(raw_dir):
        raise ValueError(f'Symlink không trỏ vào data/raw: {p} -> {target}')
    assert target.exists(), f'Symlink bị hỏng: {p} -> {target}'
    link_manifest.append({
        'link': p.relative_to(REPO_DIR).as_posix(),
        'target': target.relative_to(REPO_DIR).as_posix(),
        'is_dir': target.is_dir(),
    })

# Kiểm tra thời gian/không gian trước khi ghi archive.
needed_sources = {f'VisDrone2019-MOT-{s}' for s in SPLITS}
if 'motdet' in VARIANTS or 'det' in VARIANTS:
    needed_sources.add('VisDrone2019-DET-train')
if 'det' in VARIANTS:
    needed_sources.update(f'VisDrone2019-DET-{s}' for s in SPLITS)
size_raw = sum(p.stat().st_size for root in (sources[k] for k in needed_sources)
               for folder, _, filenames in os.walk(root)
               for name in filenames if (p:=Path(folder)/name).is_file())
size_processed = sum((Path(folder) / f).stat().st_size
                     for sub in ('processed', 'splits')
                     for folder, dirs, files in os.walk(REPO_DIR / 'data' / sub)
                     for f in files if not (Path(folder) / f).is_symlink())
free_space = shutil.disk_usage(WORK).free
estimated = size_raw + size_processed
print('Ảnh gốc + nhãn raw:', round(size_raw/1024**3, 2), 'GiB')
print('Nhãn đã chuyển đổi:', round(size_processed/1024**3, 2), 'GiB (trước nén)')
print('Free output disk:', round(free_space/1024**3, 2), 'GiB')
print('Symlinks sẽ khôi phục:', len(link_manifest))
if estimated > free_space * 0.92:
    raise RuntimeError(
        'Không đủ working disk an toàn để tạo ZIP tự chứa. '
        'Hãy giải phóng dung lượng Kaggle hoặc chọn runtime có nhiều disk hơn.'
    )


Ảnh gốc + nhãn raw: 13.07 GiB
Nhãn đã chuyển đổi: 0.86 GiB (trước nén)
Free output disk: 18.09 GiB
Symlinks sẽ khôi phục: 100328


In [8]:
BUNDLE_PREFIX = 'uav_tracking_mot_det'
zip_count = 0
archive_bytes = 0
start = time.monotonic()

def zip_rel_file(zf: zipfile.ZipFile, local: Path, relative: Path):
    global zip_count, archive_bytes
    suffix = local.suffix.lower()
    compression = zipfile.ZIP_STORED if suffix in {'.jpg','.jpeg','.png','.webp','.mp4','.zip'} else zipfile.ZIP_DEFLATED
    zf.write(local, arcname=f'{BUNDLE_PREFIX}/{relative.as_posix()}', compress_type=compression)
    zip_count += 1
    archive_bytes += local.stat().st_size

with zipfile.ZipFile(OUTPUT_ZIP, 'w', allowZip64=True, compression=zipfile.ZIP_STORED) as zf:
    # Chỉ đóng gói raw datasets thực sự dùng trong các output đã chọn.
    # Khi có MOT, prepare.py all còn xuất cả GT MOTChallenge.
    needed = set()
    if 'mot' in VARIANTS or 'motdet' in VARIANTS:
        needed.update(f'VisDrone2019-MOT-{s}' for s in SPLITS)
    if 'motdet' in VARIANTS or 'det' in VARIANTS:
        needed.add('VisDrone2019-DET-train')
    if 'det' in VARIANTS:
        needed.update(f'VisDrone2019-DET-{s}' for s in SPLITS)
    for key, src in sorted(sources.items()):
        if key not in needed:
            continue
        for folder, dirs, files in os.walk(src):
            base = Path(folder)
            for name in sorted(files):
                local = base/name
                if local.is_symlink():
                    raise ValueError(f'Raw input có symlink không kỳ vọng: {local}')
                relative = Path('data/raw') / key / local.relative_to(src)
                zip_rel_file(zf, local, relative)

    # Đóng gói các output thực của prepare.py, bỏ qua link (sẽ dựng lại bằng helper).
    for kind in ('processed','splits'):
        directory = REPO_DIR/'data'/kind
        if not directory.exists():
            continue
        for folder, dirs, files in os.walk(directory, followlinks=False):
            dirs[:] = [d for d in dirs if not (Path(folder)/d).is_symlink()]
            for name in sorted(files):
                local = Path(folder)/name
                if not local.is_symlink():
                    zip_rel_file(zf, local, local.relative_to(REPO_DIR))

    provenance = {
        'source_kaggle_datasets': 'Two Kaggle inputs: MOT (previous dataset) and DET (new dataset)',
        'source_folders': sorted(sources),
        'repository': REPO_URL,
        'repository_sha': REPO_SHA,
        'variants': VARIANTS,
        'conversion': 'data/tools/prepare.py (unmodified repository source)',
        'check': 'data/tools/check.py (unmodified repository source)',
        'images': 'Each raw image archived once; symlink layouts restored on extraction.',
    }
    zf.writestr(f'{BUNDLE_PREFIX}/provenance.json', json.dumps(provenance, ensure_ascii=False, indent=2))
    zf.writestr(f'{BUNDLE_PREFIX}/links_manifest.json', json.dumps({'links':link_manifest}, ensure_ascii=False, indent=2))
    zf.writestr(f'{BUNDLE_PREFIX}/restore_links.py', RESTORE_SCRIPT)
    zf.writestr(f'{BUNDLE_PREFIX}/README.txt', '''VISDRONE CONVERTED DATASET BUNDLE

1) Extract this ZIP to a writable directory (e.g. /kaggle/working).
2) Run: python uav_tracking_mot_det/restore_links.py
3) For YOLO11 training, use:
   data=/kaggle/working/uav_tracking_mot_det/data/processed/yolo/visdrone.yaml
   or yolo_det/visdrone.yaml when using DET.
4) RF-DETR: data/processed/rfdetr[_det|_motdet]/ has train/valid/test + COCO JSON.
5) COCO: data/processed/coco[_det|_motdet]/*.json
6) MOTChallenge: data/processed/mot/ (only when MOT exists).

Images are stored ONCE in data/raw; original repo layout symlinks are restored
by restore_links.py. The ZIP is not a native directly-trained YOLO folder before
extracting/restoring it. `path:` in visdrone.yaml is patched by restore_links.py.

Check copyright and distribution terms of the original VisDrone dataset before
publishing a new public Kaggle Dataset. Attribute the original dataset authors.
''')
    zip_count += 4

elapsed = time.monotonic()-start
print(f'ZIP DONE: {OUTPUT_ZIP}')
print(f'Contents: {zip_count:,} entries | logical input size: {archive_bytes/1024**3:.2f} GiB')
print(f'ZIP size: {OUTPUT_ZIP.stat().st_size/1024**3:.2f} GiB | time: {elapsed/60:.1f} min')


ZIP DONE: /kaggle/working/visdrone_mot_det_converted.zip
Contents: 101,420 entries | logical input size: 13.93 GiB
ZIP size: 13.18 GiB | time: 6.1 min


## 7. Kiểm tra ZIP và tạo Kaggle Dataset mới

In [9]:
with zipfile.ZipFile(OUTPUT_ZIP,'r') as zf:
    assert zf.testzip() is None, 'Archive lỗi CRC / file hỏng'
    names = set(zf.namelist())
    need = [
        'uav_tracking_mot_det/links_manifest.json',
        'uav_tracking_mot_det/restore_links.py',
        'uav_tracking_mot_det/provenance.json',
    ]
    assert all(p in names for p in need), 'Thiếu metadata/restore helper'
    assert any('/data/processed/' in name for name in names), 'Thiếu nhãn đã chuyển đổi'

print('PASS - ZIP đọc được, không lỗi CRC')
print('Output:', OUTPUT_ZIP)
print('Dung lượng:',round(OUTPUT_ZIP.stat().st_size / 1024**3,3),'GiB')
print('\\nTiếp theo: Save Version của notebook -> Output -> tạo Dataset mới từ file ZIP này.')


PASS - ZIP đọc được, không lỗi CRC
Output: /kaggle/working/visdrone_mot_det_converted.zip
Dung lượng: 13.181 GiB
\nTiếp theo: Save Version của notebook -> Output -> tạo Dataset mới từ file ZIP này.


## 8. Dọn file tạm và giữ ZIP

Chỉ xóa dữ liệu trong `/kaggle/working/uav-tracking` và thư mục DET đã giải nén, không xóa gì trong `/kaggle/input`.


In [10]:
# Xóa dữ liệu tạm để Notebook Output chỉ chứa ZIP thay vì cả repo/ảnh đã giải nén.
# Các thư mục này nằm trong /kaggle/working, không đụng đến /kaggle/input.
if CLEANUP_WORKING:
    for tmpdir in (REPO_DIR, EXTRACT_DIR):
        if tmpdir.exists():
            shutil.rmtree(tmpdir)
            print('Đã dọn:', tmpdir)
    print('ZIP cuối cùng:', OUTPUT_ZIP)


Đã dọn: /kaggle/working/uav-tracking
ZIP cuối cùng: /kaggle/working/visdrone_mot_det_converted.zip
